Load Packages

In [42]:
import pandas as pd
import json
from pathlib import Path
from collections import Counter


Load data from directory

In [22]:
file_path = Path("Data/raw/preprocessed_capstone2025.json")
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.json_normalize(data, sep="_")  # Flattens nested structure


In [14]:
print(df.columns)

Index(['1533924.xml_schema', '1533924.xml_dokument_art',
       '1533924.xml_dokument_typ', '1533924.xml_dokument_nummer',
       '1533924.xml_bibliographische-angaben_institution',
       '1533924.xml_bibliographische-angaben_aktenzeichen',
       '1533924.xml_bibliographische-angaben_datum',
       '1533924.xml_bibliographische-angaben_fundstelle_kuerzel',
       '1533924.xml_bibliographische-angaben_vorinstanz',
       '1533924.xml_bibliographische-angaben_norm_kuerzel',
       ...
       '1534093.xml_bibliographische-angaben_aktenzeichen',
       '1534093.xml_bibliographische-angaben_datum',
       '1534093.xml_bibliographische-angaben_fundstelle_kuerzel',
       '1534093.xml_bibliographische-angaben_vorinstanz',
       '1534093.xml_bibliographische-angaben_norm_kuerzel',
       '1534093.xml_text_titel', '1534093.xml_text_orientierungssatz',
       '1534093.xml_text_entscheidungsinhalt_gruende_gruende',
       '1534093.xml_allgemeine-angaben_rthema',
       '1534093.xml_interne-ang

In [29]:
# Flatten the JSON into a structured DataFrame
# Text is missing to reduce file size. has to be added later

records = []
for doc_id, content in data.items():
    base = {
        "doc_id": doc_id.removesuffix(".xml"),
        "schema": content.get("schema"),
        "dokument_art": content.get("dokument_art", [None])[0],
        "dokument_typ": content.get("dokument_typ", [None])[0],
        "institution": content.get("bibliographische-angaben", {}).get("institution", [None])[0],
        "aktenzeichen": content.get("bibliographische-angaben", {}).get("aktenzeichen", [None])[0],
        "datum": content.get("bibliographische-angaben", {}).get("datum", [None])[0],
        "fundstelle": "; ".join(content.get("bibliographische-angaben", {}).get("fundstelle_kuerzel", [])),
        "vorinstanz": "; ".join(content.get("bibliographische-angaben", {}).get("vorinstanz", [])),
        "norm_vorinstanz_aktenzeichen": "; ".join(content.get("bibliographische-angaben", {}).get("norm_vorinstanz_aktenzeichen", [])),
        "norm_kuerzel": "; ".join(content.get("bibliographische-angaben", {}).get("norm_kuerzel", [])),
        "titel": content.get("text", {}).get("titel", [None])[0],
        "rthema": "; ".join(content.get("allgemeine-angaben", {}).get("rthema", [])),
        "quelle": content.get("interne-angaben", {}).get("quelle", [None])[0]
    }
    records.append(base)

# Create a DataFrame
df2 = pd.DataFrame(records)

# Show a summary of the dataset
print("Data Summary:")
print(df2.info())

# Analyse unique values per column
print("\nUnique values per column:")
for col in df2.columns:
    print(f"{col}: {df2[col].nunique()}")

# Optional: Save to CSV for exploration
output_csv = file_path.with_suffix(".csv")
df2.to_csv(output_csv, index=False)
print(f"\nStructured summary saved to: {output_csv}")


Data Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53730 entries, 0 to 53729
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   doc_id                        53730 non-null  object
 1   schema                        53730 non-null  object
 2   dokument_art                  53730 non-null  object
 3   dokument_typ                  53730 non-null  object
 4   institution                   45474 non-null  object
 5   aktenzeichen                  45472 non-null  object
 6   datum                         53730 non-null  object
 7   fundstelle                    53730 non-null  object
 8   vorinstanz                    53730 non-null  object
 9   norm_vorinstanz_aktenzeichen  53730 non-null  object
 10  norm_kuerzel                  53730 non-null  object
 11  titel                         53556 non-null  object
 12  rthema                        53730 non-null  object
 13  qu

In [43]:
for col in df2.columns:
    if col not in ["doc_id", "norm_vorinstanz_aktenzeichen", "vorinstanz", "titel", "datum", "aktenzeichen", "rthema"]:
        print(f"Value counts for column: {col}")
        print(df2[col].value_counts())
        print("\n" + "-"*40 + "\n")

# Flatten all comma-separated values into a single list
all_rthemas = df2['rthema'].dropna().str.split(',').sum()
# Strip whitespace and count occurrences
rthema_counts = Counter(r.strip() for r in all_rthemas)

rthema_counts_df = pd.DataFrame.from_dict(rthema_counts, orient='index', columns=['count']).sort_values(by='count', ascending=False)

print(rthema_counts_df)

Value counts for column: schema
schema
rspvwa         45474
allgemeines     8256
Name: count, dtype: int64

----------------------------------------

Value counts for column: dokument_art
dokument_art
Beschluss      23739
Urteil         21735
Aufsatz         6341
Anmerkung       1617
Kurzbeitrag      273
Erwiderung        19
Übersicht          6
Name: count, dtype: int64

----------------------------------------

Value counts for column: dokument_typ
dokument_typ
Obere Rechtsprechung     45467
Literatur                 8256
Untere Rechtsprechung        7
Name: count, dtype: int64

----------------------------------------

Value counts for column: institution
institution
Bundesgerichtshof    45474
Name: count, dtype: int64

----------------------------------------

Value counts for column: fundstelle
fundstelle
                                                                                                                                 19347
GRUR-RR-2010-0407                          